In [1]:
import math

import torch
from torch import nn
from d2l import torch as d2l

In [2]:
# Position-Wise Feed-Forward Network
class PositionWiseFFN(nn.Module):
    
    def __init__(
        self,
        ffn_num_hiddens: int,
        ffn_num_outputs: int,
    ) -> None:
        super().__init__()
        
        self.dense1 = nn.LazyLinear(
            out_features=ffn_num_hiddens,
        )
        
        self.relu = nn.ReLU()
        
        self.dense2 = nn.LazyLinear(
            out_features=ffn_num_outputs
        )
        
    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        
        return self.dense2(
            self.relu(
                self.dense1(X)
            )
        )
        
        
# AddNorm
class AddNorm(nn.Module):
    
    def __init__(
        self,
        norm_shape: int,
        dropout: float,
    ) -> None:
        super().__init__()

        self.dropout = nn.Dropout(
            p=dropout,
        )

        self.layer_norm = nn.LayerNorm(
            normalized_shape=norm_shape,
        )
        
    def forward(
        self,
        X: torch.Tensor,
        Y: torch.Tensor,
    ) -> torch.Tensor:
        
        if X.shape != Y.shape:
            raise ValueError(
                "X and Y must have the same shape."
            )

        return self.layer_norm(
            X + self.dropout(Y)
        )

In [3]:
# Decoder State Type

DecoderState = tuple[
    torch.Tensor,               # Encoder Output
    torch.Tensor | None,        # Encoder Valid Length
    list[torch.Tensor | None],  # Block별 Decoder cache
]

In [4]:
# Transformer Decoder Block

class TransformerDecoderBlock(nn.Module):
    
    def __init__(
        self,
        num_hiddens: int,
        ffn_num_hiddens: int,
        num_heads: int,
        dropout: float,
        block_index: int,
    ) -> None:
        super().__init__()
        
        self.block_index = block_index
        
        # Masked Decoder Self-Attention
        self.self_attention = d2l.MultiHeadAttention(
            num_hiddens=num_hiddens,
            num_heads=num_heads,
            dropout=dropout,
        )
        
        self.add_norm1 = AddNorm(
            norm_shape=num_hiddens,
            dropout=dropout,
        )
        
        # Encoder-Decoder Attention
        self.cross_attention = d2l.MultiHeadAttention(
            num_hiddens=num_hiddens,
            num_heads=num_heads,
            dropout=dropout,
        )
        
        self.add_norm2 = AddNorm(
            norm_shape=num_hiddens,
            dropout=dropout,
        )
        
        self.ffn = PositionWiseFFN(
            ffn_num_hiddens=ffn_num_hiddens,
            ffn_num_outputs=num_hiddens,
        )
        
        self.add_norm3 = AddNorm(
            norm_shape=num_hiddens,
            dropout=dropout,
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
        state: DecoderState,
    ) -> tuple[
        torch.Tensor,
        DecoderState,
    ]:
        
        (
            encoder_outputs,
            encoder_valid_lens,
            cache,
        ) = state

        cached_key_values = cache[
            self.block_index
        ]
        
        # Training: 
        # cache가 None이므로 현재 전체 Target Sequence를 사용
        if cached_key_values is None:
            key_values = X

        # Prediction:
        # 이전 step의 cache 뒤에 현재 token을 연결
        else: key_values = torch.cat(
            (
                cached_key_values,
                X,
            ),
            dim=1,
        )
        
        cache[
            self.block_index
        ] = key_values
        
        
        # Training에서는 전체 Target Sequence가 주어지므로
        # 미래 position을 가리는 Causal Mask가 필요하다.
        if self.training:
            
            (
                batch_size,
                num_steps,
                _,
            ) = X.shape
            
            # Shape: [B, T_tgt]
            decoder_valid_lens: (
                torch.Tensor | None
            ) = torch.arange(
                1,
                num_steps + 1,
                device=X.device,
            ).repeat(
                batch_size,
                1,
            )
            
        # Prediction에서는 cache에 미래 token이 존재하지 않는다.
        else:
            decoder_valid_lens = None
            
        
        
        # 1) Masked Decoder Self-Attention
        # Query : 현재 Decoder Input
        # Key   : 현재까지의 Decoder cache
        # Value : 현재까지의 Decoder cache
        self_attention_output = self.self_attention(
            queries=X,
            keys=key_values,
            values=key_values,
            valid_lens=decoder_valid_lens,
        )
        
        Y = self.add_norm1(
            X,
            self_attention_output,
        )
        
        # 2) Encoder–Decoder Attention
        # Query : Decoder의 중간 Output
        # Key   : Encoder Output
        # Value : Encoder Output
        cross_attention_output = self.cross_attention(
            queries=Y,
            keys=encoder_outputs,
            values=encoder_outputs,
            valid_lens=encoder_valid_lens,
        )
        
        Z = self.add_norm2(
            Y,
            cross_attention_output,
        )
        
        # 3) Position-Wise FFN
        return (
            self.add_norm3(
                Z,
                self.ffn(Z),
            ),
            state,
        )

In [5]:
# Decoder Block과 Causal Mask 검증

batch_size = 2
num_steps = 100
num_hiddens = 24

num_heads = 8
ffn_num_hiddens = 48


decoder_block = TransformerDecoderBlock(
    num_hiddens=num_hiddens,
    ffn_num_hiddens=ffn_num_hiddens,
    num_heads=num_heads,
    dropout=0.5,
    block_index=0,
)

# Causal Mask를 생성하도록 Training Mode를 사용
decoder_block.train()


# Decoder Input:
# [B, T_tgt, D]
X = torch.ones(
    (
        batch_size,
        num_steps,
        num_hiddens,
    )
)

# Encoder Output:
# [B, T, H]
encoder_outputs = torch.ones(
    (
        batch_size,
        num_steps,
        num_hiddens,
    )
)

encoder_valid_lens = torch.tensor([
    3,
    2,
])


state: DecoderState = (
    encoder_outputs,
    encoder_valid_lens,
    [None],
)


with torch.no_grad():
    (
        decoder_output,
        updated_state,
    ) = decoder_block(
        X,
        state,
    )


print(
    "Decoder input:",
    tuple(X.shape),
)

print(
    "Decoder output:",
    tuple(decoder_output.shape),
)


d2l.check_shape(
    decoder_output,
    X.shape,
)


cached_values = updated_state[2][0]

if not isinstance(
    cached_values,
    torch.Tensor,
):
    raise RuntimeError(
        "Decoder cache was not created."
    )


print(
    "Cached key-values:",
    tuple(cached_values.shape),
)


d2l.check_shape(
    cached_values,
    X.shape,
)


raw_attention_weights = getattr(
    decoder_block.self_attention.attention,
    "attention_weights",
    None,
)

if not isinstance(
    raw_attention_weights,
    torch.Tensor,
):
    raise RuntimeError(
        "Self-attention weights were not computed."
    )


# [B × Head, Query, Key]
# → [B, Head, Query, Key]
attention_weights = raw_attention_weights.reshape(
    batch_size,
    num_heads,
    num_steps,
    num_steps,
)


print(
    "Self-attention weights:",
    tuple(attention_weights.shape),
)


# Query Position 0은 Key Position 0만 볼 수 있다.
assert torch.count_nonzero(
    attention_weights[
        :,
        :,
        0,
        1:,
    ]
).item() == 0


# Query Position 1은 Key Position 0과 1만 볼 수 있다.
assert torch.count_nonzero(
    attention_weights[
        :,
        :,
        1,
        2:,
    ]
).item() == 0


print(
    "Causal mask validation passed."
)

Decoder input: (2, 100, 24)
Decoder output: (2, 100, 24)
Cached key-values: (2, 100, 24)
Self-attention weights: (2, 8, 100, 100)
Causal mask validation passed.


In [6]:
# Transformer Decoder

class TransformerDecoder(d2l.AttentionDecoder):
    
    def __init__(
        self,
        vocab_size: int,
        num_hiddens: int,
        ffn_num_hiddens: int,
        num_heads: int,
        num_blocks: int,
        dropout: float,
    ) -> None:
        super().__init__() 
        
        self.num_hiddens = num_hiddens
        self.num_blocks = num_blocks

        # [B, T] -> [B, T, D]
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=num_hiddens,
        )
        
        self.positional_encoding = d2l.PositionalEncoding(
            num_hiddens=num_hiddens,
            dropout=dropout,
        )
        
        self.blocks = nn.Sequential(
            *[
                TransformerDecoderBlock(
                    num_hiddens=num_hiddens,
                    ffn_num_hiddens=ffn_num_hiddens,
                    num_heads=num_heads,
                    dropout=dropout,
                    block_index=block_index,
                )
                for block_index in range(
                    num_blocks
                )
            ]
        )
        
        # [B, T, D] -> [B, T, V]
        self.output_layer = nn.LazyLinear(
            out_features=vocab_size,
        )
        
        self._attention_weights: list[
            list[torch.Tensor]
        ] = [
            [],
            [],
        ]
        
        
        
    def init_state(
        self,
        encoder_outputs: torch.Tensor,
        encoder_valid_lens: torch.Tensor | None,
        *args: object,
    ) -> DecoderState:
        
        return (
            encoder_outputs,
            encoder_valid_lens,
            [
                None
                for _ in range(
                    self.num_blocks
                )
            ],
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
        state: DecoderState,
    ) -> tuple[
        torch.Tensor,
        DecoderState,
    ]:
        # Token Embedding: [B, T] -> [B, T, D]
        X = (
            self.embedding(X)
            * math.sqrt(
                self.num_hiddens
            )
        )
        
        # Positional Encoding: Token 순서 정보 추가
        X = self.positional_encoding(X)
        
        
        self_attention_weights: list[
            torch.Tensor
        ] = []
        
        cross_attention_weights: list[
            torch.Tensor
        ] = []
        
        
        for block in self.blocks:
            if not isinstance(
                block,
                TransformerDecoderBlock,
            ):
                raise TypeError(
                    "Every decoder block must be "
                    "a TransformerDecoderBlock."
                )
            
            X, state = block(
                X,
                state,
            )
            
            self_weights = getattr(
                block.self_attention.attention,
                "attention_weights",
                None,
            )
            
            cross_weights = getattr(
                block.cross_attention.attention,
                "attention_weights",
                None,
            )
            
            if not isinstance(
                self_weights,
                torch.Tensor,
            ):
                raise RuntimeError(
                    "Decoder self-attention weights "
                    "were not computed."
                )
            
            if not isinstance(
                cross_weights,
                torch.Tensor,
            ):
                raise RuntimeError(
                    "Cross-attention weights "
                    "were not computed."
                )
            
            self_attention_weights.append(
                self_weights
            )
            
            cross_attention_weights.append(
                cross_weights
            )
        
        
        self._attention_weights = [
            self_attention_weights,
            cross_attention_weights,
        ]
        
        # 각 Target Position에서 vocab 전체의 Logit 계산
        logits = self.output_layer(X)
        
        return logits, state
    
    
    @property
    def attention_weights(
        self,
    ) -> list[list[torch.Tensor]]:
        
        return self._attention_weights

In [7]:
# Transformer Decoder 검증

vocab_size = 100
num_hiddens = 24
ffn_num_hiddens = 48
num_heads = 4
num_blocks = 2

batch_size = 2
target_steps = 5
source_steps = 7


decoder = TransformerDecoder(
    vocab_size=vocab_size,
    num_hiddens=num_hiddens,
    ffn_num_hiddens=ffn_num_hiddens,
    num_heads=num_heads,
    num_blocks=num_blocks,
    dropout=0.0,
)

# Training Mode에서 Causal Mask 생성
decoder.train()


# Target Token
# Shape: [B, T_tgt]
target_tokens = torch.ones(
    (
        batch_size,
        target_steps,
    ),
    dtype=torch.long,
)

# Encoder Output
# Shape: [B, T_src, D]
encoder_outputs = torch.randn(
    (
        batch_size,
        source_steps,
        num_hiddens,
    )
)

encoder_valid_lens = torch.tensor([
    source_steps,
    5,
])


state = decoder.init_state(
    encoder_outputs,
    encoder_valid_lens,
)


with torch.no_grad():
    logits, updated_state = decoder(
        target_tokens,
        state,
    )


print(
    "Target tokens:",
    tuple(target_tokens.shape),
)

print(
    "Encoder outputs:",
    tuple(encoder_outputs.shape),
)

print(
    "Decoder logits:",
    tuple(logits.shape),
)


d2l.check_shape(
    logits,
    (
        batch_size,
        target_steps,
        vocab_size,
    ),
)


self_attention_weights = (
    decoder.attention_weights[0][0]
).reshape(
    batch_size,
    num_heads,
    target_steps,
    target_steps,
)

cross_attention_weights = (
    decoder.attention_weights[1][0]
).reshape(
    batch_size,
    num_heads,
    target_steps,
    source_steps,
)


print(
    "Self-attention weights:",
    tuple(
        self_attention_weights.shape
    ),
)

print(
    "Cross-attention weights:",
    tuple(
        cross_attention_weights.shape
    ),
)


# 첫 Target Token은 미래 Target Token을 볼 수 없다.
assert torch.count_nonzero(
    self_attention_weights[
        :,
        :,
        0,
        1:,
    ]
).item() == 0


# 두 번째 example의 유효한 Source Length는 5다.
# 따라서 Source Position 5 이후의 Weight는 0이다.
assert torch.count_nonzero(
    cross_attention_weights[
        1,
        :,
        :,
        5:,
    ]
).item() == 0


for block_index, cached_values in enumerate(
    updated_state[2]
):
    if not isinstance(
        cached_values,
        torch.Tensor,
    ):
        raise RuntimeError(
            f"Block {block_index} cache was not created."
        )
    
    print(
        f"Block {block_index} cache:",
        tuple(cached_values.shape),
    )


print(
    "Transformer Decoder validation passed."
)

Target tokens: (2, 5)
Encoder outputs: (2, 7, 24)
Decoder logits: (2, 5, 100)
Self-attention weights: (2, 4, 5, 5)
Cross-attention weights: (2, 4, 5, 7)
Block 0 cache: (2, 5, 24)
Block 1 cache: (2, 5, 24)
Transformer Decoder validation passed.
